# Урок 10 - Агенти ШІ в продуктивному середовищі

У цьому уроці ви дізнаєтесь про **шаблони для експлуатації в продуктивному середовищі** агентів ШІ з використанням Microsoft Agent Framework (Python). Ми розглядаємо:

- **Спостережуваність** — додавання вимірювання часу та логування до взаємодій агента
- **Оцінювання** — використання агента-оцінювача для оцінювання якості відповідей
- **Управління витратами** — стратегії оптимізації токенів та вибору моделі

Сценарій — це **туристичний агент**, який допомагає користувачам планувати подорожі, з доданим моніторингом та оцінюванням.


## Налаштування


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Міркування щодо продакшну

Перехід агентів ШІ від прототипів до продакшну вимагає ретельної уваги до трьох стовпів:

1. **Спостережуваність** — Вам потрібна видимість того, що робить агент, скільки часу це займає, та які інструменти він викликає. Без трасування та логування налагодження проблем у продакшні майже неможливе.

2. **Оцінювання** — Автоматизовані перевірки якості забезпечують, що відповіді агента з часом залишаються точними, повними та корисними. Агент-оцінювач може оцінювати відповіді за визначеними критеріями.

3. **Управління витратами** — Використання токенів безпосередньо впливає на витрати. Стратегії, такі як оптимізація підказок, вибір моделі та кешування, допомагають тримати витрати під контролем без шкоди для якості.


## Створення спостережуваного агента

Ми визначаємо інструменти для подорожей і обгортаємо виклик агента засобами вимірювання часу, щоб мати змогу відстежувати затримку. У виробничому середовищі ви інтегрували б OpenTelemetry або подібну систему трасування.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Шаблони оцінювання

Поширений виробничий підхід — використовувати другого агента як **оцінювача**. Оцінювач виставляє оцінку відповіді основного агента за заздалегідь визначеними критеріями, такими як повнота, точність і корисність.

This enables:
- Автоматичні механізми контролю якості перед тим, як відповіді потраплять до користувачів
- Виявлення регресій при зміні підказок або моделей
- Постійний моніторинг продуктивності агента з плином часу


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Стратегії управління витратами

Контроль витрат критично важливий для виробничих агентів ШІ. Ось ключові стратегії:

| Стратегія | Опис |
|---|---|
| **Оптимізація підказок** | Тримайте системні інструкції лаконічними. Видаляйте зайвий контекст, щоб зменшити кількість вхідних токенів. |
| **Вибір моделі** | Використовуйте менші, дешевші моделі (наприклад GPT-4o-mini) для простих завдань, таких як класифікація або витягнення, а більші моделі залишайте для складніших міркувань. |
| **Кешування** | Кешуйте результати інструментів та часті запити, щоб уникнути повторних викликів API. |
| **Бюджети токенів** | Встановлюйте обмеження `max_tokens`, щоб запобігти несподівано довгим відповідям. |
| **Пакетна обробка** | Об’єднуйте кілька запитів користувача в один виклик API там, де це можливо. |

На практиці добре працює багаторівневий підхід: направляйте прості запити до швидкої, недорогої моделі, а складніші запити переадресовуйте на більш потужну.


## Підсумок

У цьому уроці ви дізналися, як:

1. **Додавати спостережуваність** до взаємодій агентів за допомогою вимірювання часу та логування, закладаючи основу для трасування та моніторингу.
2. **Оцінювати відповіді агента** автоматично за допомогою агента-оцінювача, який оцінює повноту, точність і корисність.
3. **Керувати витратами** через оптимізацію підказок, вибір моделей, кешування та бюджети токенів.

Ці виробничі підходи допомагають забезпечити надійність, вимірюваність і економічну ефективність ваших агентів ШІ в масштабі.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Відмова від відповідальності:
Цей документ було перекладено з використанням сервісу машинного перекладу Co‑op Translator (https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, зауважте, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний переклад, виконаний людиною. Ми не несемо відповідальності за будь‑які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
